# **PLAYERS SEMANTIC ATLAS**

This notebook explores the high-dimensional latent space where `TacticalBert` represents football players. This code uses **contextual embeddings**, a mathematical representations of players derived from their actions, positions and spatial context.

Inspired by "*Visualizing and Measuring the Geometry of BERT*" (Coenen et al., 2019), the code demonstrates that the model naturally clusters players by their tactical roles without being explicity trained on positions (player role).

## **SETUP and DATA LOADING**
Before starting, remember to run `generate_players_vectors.py` that creates two files in the `data/processed` directory:
- `player_embeddings_db.parquet` : contains the vectors representing the average of all last hidden states for a player across all his recorded matches.
- `player_metadata.json` : contains some informations for each player useful for the visualization (id, player_name, position_name, team_name)


In [16]:
import pandas as pd
import numpy as np
import umap
import plotly.express as px
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

cwd = Path.cwd()
proj_root = cwd
while proj_root != proj_root.parent and not (proj_root / "src").is_dir():
    proj_root = proj_root.parent
if (proj_root / "src").is_dir():
    sys.path.insert(0, str(proj_root))
else:
    sys.path.insert(0, str(cwd))

from src.config import Config


OUTPUT_DB_PATH = Config.PROCESSED_DATA_DIR / "player_embeddings_db.parquet"
df = pd.read_parquet(OUTPUT_DB_PATH)
df_clean = df[
    (df['actions_count'] > 1000) & 
    (df['position'] != 'Unknown') & 
    (df['position'] != 'Unknown/Substitute')
].copy()
print(f"Number of players after analyzed: {len(df_clean)}")
df_clean.head()


Number of players after analyzed: 1602


,player_id,player_name,position,team,actions_count,embedding
0,8575,Salomon Armand Magloire Kalou,Left Wing,Hertha Berlin,2320,"[0.02273423847093062, 0.09437569772488696, 0.0..."
2,8579,Vladimír Darida,Center Attacking Midfield,Hertha Berlin,3680,"[0.006464194882126089, 0.05250622747168468, 0...."
3,8580,Sebastian Langkamp,Right Center Back,Hertha Berlin,2304,"[-0.004010309885033949, 0.08493096244835543, -..."
4,8965,Fabian Lustenberger,Left Center Back,Hertha Berlin,2830,"[0.009317420049303733, 0.04310910138220307, -0..."
5,5569,Marvin Plattenhardt,Left Back,Hertha Berlin,3249,"[0.1512384581649593, -0.036353357063458616, -0..."


## **DIMENSIONALITY REDUCTION (UMAP)**
In order to make navigable the 128-dimensional space player embeddings, the code uses **UMAP (Uniform Manifold Approximation and Projection)**. This algorithm is particularly effective at preserving both local relationships (similar players) and global topological structure (different roles).

In [11]:
#Mapping specific positions to macro-roles for better visualization and analysis.
role_mapping = {
    'Goalkeeper': 'Goalkeeper',
    
    'Center Back': 'Central Defender',
    'Left Center Back': 'Central Defender',
    'Right Center Back': 'Central Defender',
    
    'Left Back': 'Fullback/Wingback',
    'Right Back': 'Fullback/Wingback',
    'Left Wing Back': 'Fullback/Wingback',
    'Right Wing Back': 'Fullback/Wingback',
    
    'Center Defensive Midfield': 'Midfielder',
    'Left Defensive Midfield': 'Midfielder',
    'Right Defensive Midfield': 'Midfielder',
    'Left Center Midfield': 'Midfielder',
    'Right Center Midfield': 'Midfielder',
    'Center Midfield': 'Midfielder',

    'Center Attacking Midfield': 'Offensive Midfielder',
    'Left Attacking Midfield': 'Offensive Midfielder',
    'Right Attacking Midfield': 'Offensive Midfielder',
    
    'Left Midfield': 'Winger',
    'Right Midfield': 'Winger',
    'Left Wing': 'Winger',
    'Right Wing': 'Winger',

    'Center Forward': 'Attacker',
    'Left Center Forward': 'Attacker',
    'Right Center Forward': 'Attacker',
    'Second Striker': 'Attacker'
}

df_clean['macro_position'] = df_clean['position'].map(role_mapping).fillna('Other')

print("Computing UMAP projections...")
X = np.stack(df_clean['embedding'].values)
reducer = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.1, metric='cosine', random_state=42)
projections = reducer.fit_transform(X)

df_clean['x'] = projections[:, 0]
df_clean['y'] = projections[:, 1]
print("UMAP projections computed successfully.")

Computing UMAP projections...
UMAP projections computed successfully.


## **INTERACTIVE SEMANTIC ATLAS**
The following map represents the tactical atlas of the model. The model doesn't know the tactical roles of the players and the colored labels are assigned only in this moment. Players clustered together share similar action patterns, spatial locations and pressure handling.

In [15]:
fig = px.scatter(
    df_clean, x='x', y='y', color='macro_position', hover_name='player_name',
    hover_data={'x': False, 'y': False, 'position': True, 'team': True, 'actions_count': True},
    title='<b>TacticalBERT Player Atlas</b>',
    template='plotly_dark', width=800, height=600,
    category_orders={"macro_position": [
        "Goalkeeper", "Central Defender", "Fullback/Wingback", 
        "Midfielder", "Offensive Midfielder", "Winger", "Attacker"
    ]}
)

fig.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=0.5, color='white')))
fig.show()

Consistent with the *Geometry of Word Senses* observed in linguistic BERT models, the atlas reflects a clear semantic separation between specialized roles. 

Goalkeepers form a isolated cluster, indicating a unique distribution of actions that rarely overlaps with field players. In contrast, the map shows a high degree of local density and overlap for the midfielders, offensive midfielders, wingers and attackers. This captures the tactical fluidity of modern football where these roles often interchange.

## **SEMANTIC SIMILARITY SEARCH**
Since players are represented as vectors, it's possible to use **Cosine Similarity** to quantify how tactically similar two players are. This allows for automated scouting: finding a replacement for a specific player by searching for their nearest neighbors in the latent space.

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_players(player_name, top_n=5):
    if player_name not in df_clean['player_name'].values:
        return f"Player '{player_name}' not found in the database."
    
    # Get target player embedding
    target_idx = df_clean[df_clean['player_name'] == player_name].index[0]
    target_vector = df_clean.loc[target_idx, 'embedding'].reshape(1, -1)
    
    # Compute similarity with all others
    all_vectors = np.stack(df_clean['embedding'].values)
    similarities = cosine_similarity(target_vector, all_vectors).flatten()
    
    results_df = df_clean.copy()
    results_df['similarity'] = similarities
    
    return results_df[results_df['player_name'] != player_name].sort_values(by='similarity', ascending=False).head(top_n)[
        ['player_name', 'position', 'team', 'similarity']
    ]

target_player = df_clean[df_clean['player_name'] == 'Lionel Andr\u00e9s Messi Cuccittini']['player_name'].iloc[0] # You can change this to "Lionel Messi" or any available name
print(f"Top 5 players most similar to {target_player}:")
find_similar_players(target_player)

Top 5 players most similar to Lionel Andrés Messi Cuccittini:


,player_name,position,team,similarity
2436,Neymar da Silva Santos Junior,Left Wing,Barcelona,0.997488
583,Eden Hazard,Left Wing,Chelsea,0.996472
105,Douglas Costa de Souza,Left Midfield,Bayern Munich,0.996100
561,Alexis Alejandro Sánchez Sánchez,Left Wing,Arsenal,0.995810
1160,Lucas Rodrigues Moura da Silva,Right Wing,Paris Saint-Germain,0.995533


The high cosine similarity between Messi, Neymar and Hazard confirms that the model has internalized technical synonyms. These players share tactical signature in the latent space, defined by high frequency dribbling, specific spatial heatmaps, and creative passing under pressure

In [25]:
target_player = df_clean[df_clean['player_name'] == 'Gianluigi Buffon']['player_name'].iloc[0] # You can change this to "Lionel Messi" or any available name
print(f"Top 5 players most similar to {target_player}:")
find_similar_players(target_player)

Top 5 players most similar to Gianluigi Buffon:


,player_name,position,team,similarity
1168,Kevin Trapp,Goalkeeper,Paris Saint-Germain,0.997427
1027,Stéphane Ruffier,Goalkeeper,Saint-Étienne,0.997069
1062,Anthony Lopes,Goalkeeper,Lyon,0.996940
1602,Gianluigi Donnarumma,Goalkeeper,AC Milan,0.996912
1012,Cédric Carrasso,Goalkeeper,Bordeaux,0.996522
